# PhenoBench TensorFlow Object Detection Dataset Export

This notebook converts the :contentReference[oaicite:0]{index=0} dataset into TensorFlow Object Detection API compatible artifacts.

Generated outputs include:
- TFRecord datasets
- train/validation/test splits
- TensorFlow label maps
- representative dataset indices for INT8 quantization
- dataset metadata for reproducibility

The resulting artifacts are intended for:
- TensorFlow Object Detection API training
- post-training quantization (PTQ)
- quantization-aware training (QAT)
- TensorFlow Lite deployment

## Environment Setup

This notebook targets:
- Python 3.10
- TensorFlow-compatible preprocessing pipeline
- Kaggle notebook runtime

For reproducibility, the exact repository commit is pinned.

In [ ]:
!python --version

In [ ]:
!pip install \
  git+https://github.com/frdiener/agri-vision-edge.git@d37651c425f4308cdee2e88891636bcc73eb31c0 \
  --no-deps

In [ ]:
from pathlib import Path
import json

from agri_vision_edge.data import (
    PhenoBench,
    split_indices,
    build_record,
    build_rep_indices,
    write_label_map,
)

SEED = 42
IMAGE_SIZE = 320

DATASET_ROOT = Path(
    "../input/datasets/freimutdiener/phenobench-raw-dataset-v1-1-0/PhenoBench"
)

assert DATASET_ROOT.exists()

## Dataset Loading

We load the original PhenoBench semantic and instance annotations
and convert them into object detection targets.

In [ ]:
train_dataset = PhenoBench(
    root=DATASET_ROOT,
    split="train",
    target_types=["semantics", "plant_instances"],
    ignore_partial=False,
)

val_dataset = PhenoBench(
    root=DATASET_ROOT,
    split="val",
    target_types=["semantics", "plant_instances"],
    ignore_partial=False,
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

## Validation/Test Split

The original validation split is subdivided into:
- validation
- held-out test

using a deterministic random seed.

In [ ]:
val_idx, test_idx = split_indices(
    len(val_dataset),
    val_ratio=0.5,
    seed=SEED,
)

with open("val_test_split.json", "w") as f:
    json.dump(
        {
            "val": val_idx,
            "test": test_idx,
        },
        f,
    )

## TFRecord Export

The segmentation annotations are converted into object detection targets:

- semantic masks determine class labels
- instance masks determine object instances
- bounding boxes are extracted from connected components

Generated datasets:
- `train.record`
- `val.record`
- `test.record`
- `true_eval.record`

Images are resized to:
- 320 × 320

In [ ]:
train_stats = build_record(
    "train.record",
    train_dataset,
    target_size=IMAGE_SIZE,
    with_tqdm=True,
)

true_eval_stats = build_record(
    "true_eval.record",
    val_dataset,
    target_size=IMAGE_SIZE,
    with_tqdm=True,
)

val_stats = build_record(
    "val.record",
    val_dataset,
    indices=val_idx,
    target_size=IMAGE_SIZE,
    with_tqdm=True,
)

test_stats = build_record(
    "test.record",
    val_dataset,
    indices=test_idx,
    target_size=IMAGE_SIZE,
    with_tqdm=True,
)

## Representative Dataset for INT8 Quantization

A representative calibration subset is generated for:
- post-training quantization (PTQ)
- quantization-aware training (QAT)

The exported indices ensure:
- deterministic calibration
- reproducible quantization experiments
- consistent deployment benchmarking

In [ ]:
rep_indices = build_rep_indices(
    dataset=train_dataset,
    num_samples=200,
    seed=SEED,
)

with open("rep_dataset.json", "w") as f:
    json.dump(rep_indices, f)

## Label Map

TensorFlow Object Detection API requires a label map definition.

In [ ]:
write_label_map("label_map.pbtxt")

## Dataset Metadata

We export metadata required for:
- reproducibility
- downstream training
- deployment consistency

In [ ]:
metadata = {
    "image_size": IMAGE_SIZE,
    "classes": {
        "1": "crop",
        "2": "weed",
    },
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "rep_samples": len(rep_indices),
    "split_seed": SEED,

    "train_stats": train_stats,
    "true_eval_stats": true_eval_stats,
    "val_stats": val_stats,
    "test_stats": test_stats,
}

with open("dataset_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

## Artifact Verification

Verify all required outputs exist before notebook export.

In [ ]:
artifacts = [
    "train.record",
    "val.record",
    "test.record",
    "true_eval.record",
    "label_map.pbtxt",
    "rep_dataset.json",
    "val_test_split.json",
    "dataset_metadata.json",
]

missing = [
    p for p in artifacts
    if not Path(p).exists()
]

assert not missing, f"Missing artifacts: {missing}"

print("All artifacts generated successfully.")

In [ ]:
!ls -al

In [ ]:
!cat dataset_metadata.json

## Notes

This export pipeline provides a reproducible bridge between:

- semantic/instance segmentation annotations
- TensorFlow object detection training
- edge deployment workflows

The generated artifacts are intended for:
- SSD MobileNet training
- TensorFlow Lite conversion
- INT8 quantization
- agricultural robotics research